# Model 3 - 3B: ResNet50 đóng băng

Transfer learning: ResNet50 đã train sẵn trên ImageNet bị đóng băng hoàn toàn, chỉ train head MLP.

Chạy các cell từ trên xuống: import, cấu hình, xử lý ảnh, load dữ liệu, tạo model, train/validation, test và lưu model.

In [1]:
# 1. Import và cấu hình
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report, precision_recall_fscore_support

DATA_DIR = '../../../data/VN_trash_classification_preprocessing'
CHECKPOINT_DIR = '../checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Trọng số ImageNet được tải về (hoặc đã có sẵn) trong src/model3/pretrained
torch.hub.set_dir('../pretrained/hub')

BATCH_SIZE = 64
EPOCHS = 30
PATIENCE = 6
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [2]:
# 2. Xử lý ảnh
normalize = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    normalize,
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])

In [3]:
# 3. Load dữ liệu
train_dataset = ImageFolder(DATA_DIR + '/train', transform=train_transform)
val_dataset = ImageFolder(DATA_DIR + '/val', transform=test_transform)
test_dataset = ImageFolder(DATA_DIR + '/test', transform=test_transform)

# drop_last=True: tránh batch chỉ có 1 ảnh làm BatchNorm1d của head báo lỗi
# Nạp ảnh bằng nhiều tiến trình để GPU không phải chờ CPU
loader_options = {'num_workers': 8, 'pin_memory': True, 'persistent_workers': True}
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **loader_options)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_options)

class_names = train_dataset.classes
print('Classes:', class_names)
print('Train:', len(train_dataset), '| Validation:', len(val_dataset), '| Test:', len(test_dataset))

Classes: ['Alu', 'Carton', 'Foam_box', 'Milk_box', 'Other', 'PET', 'Paper', 'Paper_cup', 'Plastic_cup']
Train: 9542 | Validation: 1685 | Test: 864


In [4]:
# 4. Tạo model: backbone ImageNet + head MLP
class TransferModel(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()  # backbone đóng băng nên luôn ở chế độ eval
        return self

    def forward(self, images):
        with torch.no_grad():
            features = self.backbone(images)
        return self.head(features)


# ResNet50 pretrained ImageNet, bỏ lớp fc cuối (2048 -> 1000) để lấy vector 2048 chiều
backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
backbone.fc = nn.Identity()
feature_dim = 2048

# Đóng băng toàn bộ backbone, chỉ train head
for parameter in backbone.parameters():
    parameter.requires_grad = False

head = nn.Sequential(
    nn.Linear(feature_dim, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(inplace=True),
    nn.Dropout(0.3),
    nn.Linear(512, len(class_names)),
)

model = TransferModel(backbone, head).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.head.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
total = sum(parameter.numel() for parameter in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')

Trainable params: 1,054,729 / 24,562,761


In [5]:
# 5. Train và Validation
def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    correct = 0
    seen = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = loss_function(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        seen += labels.size(0)

    return total_loss / seen, correct / seen


best_val_accuracy = 0.0
best_weights = None
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    val_loss, val_accuracy = run_epoch(val_loader, training=False)
    scheduler.step()

    # Sao chép trọng số sang RAM để các epoch sau không ghi đè lên bản tốt nhất.
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_weights = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
        best_epoch = epoch + 1
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f'Epoch {epoch + 1}/{EPOCHS} | '
        f'Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.2%} | '
        f'Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2%}'
    )

    # Dừng khi validation accuracy không tăng trong PATIENCE epoch liên tiếp.
    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping tại epoch {epoch + 1}')
        break

# Vòng lặp kết thúc khi model đang giữ trọng số của epoch cuối, nạp lại trọng số tốt nhất.
model.load_state_dict(best_weights)
print(f'Best epoch: {best_epoch} | Best Val Acc: {best_val_accuracy:.2%}')

Epoch 1/30 | Train Loss: 0.4418 | Train Acc: 86.13% | Val Loss: 0.2592 | Val Acc: 91.28%


Epoch 2/30 | Train Loss: 0.1801 | Train Acc: 94.69% | Val Loss: 0.2518 | Val Acc: 92.05%


Epoch 3/30 | Train Loss: 0.1051 | Train Acc: 97.02% | Val Loss: 0.2587 | Val Acc: 91.51%


Epoch 4/30 | Train Loss: 0.0769 | Train Acc: 97.88% | Val Loss: 0.2554 | Val Acc: 92.40%


Epoch 5/30 | Train Loss: 0.0517 | Train Acc: 98.73% | Val Loss: 0.2522 | Val Acc: 92.70%


...

Epoch 8/30 | Train Loss: 0.0222 | Train Acc: 99.50% | Val Loss: 0.2909 | Val Acc: 92.46%


Epoch 9/30 | Train Loss: 0.0218 | Train Acc: 99.45% | Val Loss: 0.2892 | Val Acc: 93.12%


Epoch 10/30 | Train Loss: 0.0182 | Train Acc: 99.53% | Val Loss: 0.2976 | Val Acc: 92.52%


Epoch 11/30 | Train Loss: 0.0148 | Train Acc: 99.66% | Val Loss: 0.2961 | Val Acc: 92.82%


Epoch 12/30 | Train Loss: 0.0114 | Train Acc: 99.85% | Val Loss: 0.2982 | Val Acc: 93.00%
Early stopping tại epoch 12
Best epoch: 6 | Best Val Acc: 93.23%


In [6]:
# 6. Test
model.eval()
true_labels = []
predicted_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        true_labels += labels.tolist()
        predicted_labels += outputs.argmax(dim=1).cpu().tolist()

accuracy = sum(t == p for t, p in zip(true_labels, predicted_labels)) / len(true_labels)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    true_labels, predicted_labels, average='macro', zero_division=0
)
_, _, weighted_f1, _ = precision_recall_fscore_support(
    true_labels, predicted_labels, average='weighted', zero_division=0
)

print()
print('Test Result')
print('Test Accuracy:', f'{accuracy:.2%}')
print('Macro Precision:', f'{macro_precision:.4f}')
print('Macro Recall:', f'{macro_recall:.4f}')
print('Macro F1:', f'{macro_f1:.4f}')
print('Weighted F1:', f'{weighted_f1:.4f}')
print()
print(classification_report(true_labels, predicted_labels, target_names=class_names, digits=4, zero_division=0))


Test Result
Test Accuracy: 93.98%
Macro Precision: 0.9376
Macro Recall: 0.9366
Macro F1: 0.9363
Weighted F1: 0.9397

              precision    recall  f1-score   support

         Alu     0.9600    0.9697    0.9648        99
      Carton     0.9899    0.9074    0.9469       108
    Foam_box     0.9740    0.8621    0.9146        87
    Milk_box     0.9619    1.0000    0.9806       101
       Other     0.8654    0.9000    0.8824        50
         PET     0.9412    0.9412    0.9412       102
       Paper     0.9346    0.9434    0.9390       106
   Paper_cup     0.9273    0.9714    0.9488       105
 Plastic_cup     0.8839    0.9340    0.9083       106

    accuracy                         0.9398       864
   macro avg     0.9376    0.9366    0.9363       864
weighted avg     0.9412    0.9398    0.9397       864



In [7]:
# 7. Lưu model
torch.save(model.state_dict(), CHECKPOINT_DIR + '/3b_frozen_resnet50.pt')